In [39]:
import os
import pandas as pd

# Define the path to the export folder
export_folder_path = '/content/export/'

# List all files in the export folder
all_files = os.listdir(export_folder_path)

# Filter for CSV files
csv_files = [f for f in all_files if f.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files in '{export_folder_path}':")
for f in csv_files:
    print(f)

Found 12 CSV files in '/content/export/':
export_20260603-141144.csv
export_20260621-155053.csv
export_20260603-134331.csv
export_20260603-140315.csv
export_20260621-154216.csv
export_20260603-133844.csv
export_20260621-155611.csv
export_20260603-133359.csv
export_20260603-133644.csv
export_20260603-142601.csv
export_20260603-133158.csv
export_20260621-155252.csv


Now, let's take a look at the first CSV file to understand its structure. This will help us identify how comment rows are represented.

Now, let's extract the 'Comment' column from each CSV file and combine them into a new `HSP.csv` file.

In [40]:
import re
import pandas as pd

all_comments = pd.Series(dtype=str)

if csv_files:
    for csv_file in csv_files:
        file_path = os.path.join(export_folder_path, csv_file)
        try:
            # Read the CSV file, specifically looking for the 'Comment' column
            # Using 'engine='python'' to help with delimiter detection and 'encoding' for common issues
            # Changed encoding to 'utf-8-sig' for better compatibility with non-ASCII characters
            df = pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig')

            # Check if 'Comment' column exists
            if 'Comment' in df.columns:
                # Append comments to our series, dropping any rows where 'Comment' is NaN
                all_comments = pd.concat([all_comments, df['Comment'].dropna()])
            else:
                print(f"Warning: 'Comment' column not found in {csv_file}. Skipping this file.")
        except Exception as e:
            print(f"Error reading {csv_file}: {e}. Skipping this file.")

    # Filter out comments that are primarily URLs
    url_pattern = r'http[s]?://\S+'
    initial_comment_count = len(all_comments)
    all_comments = all_comments[~all_comments.astype(str).str.contains(url_pattern, na=False)]
    print(f"Filtered out {initial_comment_count - len(all_comments)} comments containing URLs.")

    # Remove duplicate comments
    initial_unique_comment_count = len(all_comments)
    all_comments = all_comments.drop_duplicates()
    print(f"Removed {initial_unique_comment_count - len(all_comments)} duplicate comments.")

    # Save the combined comments to HSP.csv
    output_file_path = '/content/HSP.csv'
    # Saving with 'utf-8-sig' encoding to ensure external programs correctly display Bengali text
    all_comments.to_csv(output_file_path, index=False, header=['Comment'], encoding='utf-8-sig')
    print(f"Successfully combined comments into '{output_file_path}'")
    print(f"Number of combined and unique comments after filtering URLs: {len(all_comments)}")

    # Display the head of the combined comments file, reading with the same encoding
    df_hsp = pd.read_csv(output_file_path, encoding='utf-8-sig')
    print("\nHead of the generated HSP.csv file:")
    display(df_hsp.head())
else:
    print("No CSV files found to process.")

Filtered out 31 comments containing URLs.
Removed 96 duplicate comments.
Successfully combined comments into '/content/HSP.csv'
Number of combined and unique comments after filtering URLs: 1004

Head of the generated HSP.csv file:


,Comment
0,যারাই হক্বের কথা বলবে তারাই শিবির🔥
1,লুঙ্গি স্পায় 😂😂
2,দাড়িপাল্লা ⚖️
3,ভাই আপনারে Dhurandar মুভিতে দরকার ছিলো
4,❤️❤️❤️❤️🔥🔥🔥🔥


In [41]:
import emoji
import pandas as pd

# Load the HSP.csv file
output_file_path = '/content/HSP.csv'
df_hsp = pd.read_csv(output_file_path, encoding='utf-8-sig')

# List to store extracted emojis and their names
all_extracted_emojis = []

# Iterate through each comment to extract emojis
for comment in df_hsp['Comment'].dropna():
    emojis_in_comment = emoji.emoji_list(comment)
    for emo_dict in emojis_in_comment:
        e = emo_dict['emoji']
        # Get the emoji name (shortcode), removing colons and replacing underscores with spaces
        emoji_name = emoji.demojize(e).replace(':', '').replace('_', ' ')
        all_extracted_emojis.append({'Emoji': e, 'Name': emoji_name})

# Create a DataFrame from the extracted emojis
df_emojis = pd.DataFrame(all_extracted_emojis)

# Remove duplicates, keeping the first occurrence
df_emojis = df_emojis.drop_duplicates().reset_index(drop=True)

# --- New code to classify emojis as hate/non-hate ---
# Define a list of emojis that might be considered hate-related (this list can be expanded)
# This list is illustrative and based on common interpretations.
# Real-world hate speech detection is complex and context-dependent.
hate_emojis = [
    '🖕', # Middle Finger
    '🔪', # Kitchen Knife
    '💣', # Bomb
    '🔫', # Pistol
    '🤬', # Face with Symbols on Mouth
    '😠', # Angry Face
    '😡', # Pouting Face
    '😈', # Smiling Face with Horns
    '💀', # Skull
    '☠️', # Skull and Crossbones
    '💩', # Pile of Poo
    '🤢', # Nauseated Face
    '🤮', # Vomiting Face
    '🕷️', # Spider
    '🐍', # Snake
    '🐀', # Rat
    '🦍', # Gorilla (can be used in racist contexts)
    '🐷', # Pig Face (can be used as insult)
    '👹', # Ogre
    '👺'  # Goblin
]

# Add 'IsHate' column: 1 if emoji is in hate_emojis, else 0
df_emojis['IsHate'] = df_emojis['Emoji'].apply(lambda x: 1 if x in hate_emojis else 0)
# -----------------------------------------------------

# Save the DataFrame to emoji.csv
emoji_output_path = '/content/emoji_data/emoji.csv'
df_emojis.to_csv(emoji_output_path, index=False, encoding='utf-8-sig')

print(f"Successfully extracted {len(df_emojis)} unique emojis and saved to '{emoji_output_path}'")

# Display the head of the generated emoji.csv file
print("\nHead of the generated emoji.csv file with 'IsHate' column:")
display(df_emojis.head())

Successfully extracted 166 unique emojis and saved to '/content/emoji_data/emoji.csv'

Head of the generated emoji.csv file with 'IsHate' column:


,Emoji,Name,IsHate
0,🔥,fire,0
1,😂,face with tears of joy,0
2,⚖️,balance scale,0
3,❤️,red heart,0
4,👌,OK hand,0


In [42]:
if csv_files:
    # Read the first CSV file to inspect its structure
    first_csv_path = os.path.join(export_folder_path, csv_files[0])
    print(f"Reading the first CSV file: {first_csv_path}")

    # Attempt to read the CSV, treating potentially problematic lines as comments if possible.
    # Using 'comment' parameter or 'engine='python'' might be necessary depending on the actual format.
    # For now, let's try a standard read and inspect.
    try:
        df_sample = pd.read_csv(first_csv_path, sep=None, engine='python') # Use engine='python' for better delimiter inference
        display(df_sample.head())
        print("\nDataFrame Info:")
        df_sample.info()
    except Exception as e:
        print(f"Could not read {first_csv_path} with default settings. Error: {e}")
        print("Trying to read as plain text to inspect content...")
        with open(first_csv_path, 'r') as f:
            for i, line in enumerate(f):
                print(line.strip())
                if i >= 10: # Print first 10 lines for inspection
                    break
else:
    print("No CSV files found in the export folder.")

Reading the first CSV file: /content/export/export_20260603-141144.csv


,﻿,Unnamed: 1,Name,Profile ID,Date,Likes,Live video timestamp,Comment,Image URLs,Comment Permalink,Depth,Mentions,Reactions,Author Avatar,Comment Edited,Comment URL
0,1,NaN,Vatiz Gallery,61556543624128,2026-02-01 15:40:26,0,-,যারাই হক্বের কথা বলবে তারাই শিবির🔥,NaN,NaN,0,NaN,NaN,https://scontent-tpe1-1.xx.fbcdn.net/v/t39.308...,NaN,https://www.exportcomments.com/done/3f7fbc3a-6...
1,2,NaN,Sai Han,61585165719638,2026-02-01 21:01:02,0,-,লুঙ্গি স্পায় 😂😂,NaN,NaN,0,NaN,NaN,https://scontent-tpe1-1.xx.fbcdn.net/v/t39.308...,NaN,https://www.exportcomments.com/done/3f7fbc3a-6...
2,3,NaN,Najifa's World,61582320181867,2026-02-01 21:11:11,0,-,দাড়িপাল্লা ⚖️,NaN,NaN,0,NaN,NaN,https://scontent-tpe1-1.xx.fbcdn.net/v/t39.308...,NaN,https://www.exportcomments.com/done/3f7fbc3a-6...
3,4,NaN,Fahad Bin Aziz,100016705966798,2026-02-02 04:23:35,0,-,ভাই আপনারে Dhurandar মুভিতে দরকার ছিলো,NaN,NaN,0,NaN,NaN,https://scontent-tpe1-1.xx.fbcdn.net/v/t39.308...,NaN,https://www.exportcomments.com/done/3f7fbc3a-6...
4,5,NaN,Saddam for Humanity,61578976214728,2026-02-02 04:32:38,0,-,❤️❤️❤️❤️🔥🔥🔥🔥,NaN,NaN,0,NaN,NaN,https://scontent-tpe1-1.xx.fbcdn.net/v/t39.308...,NaN,https://www.exportcomments.com/done/3f7fbc3a-6...



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ﻿                     100 non-null    int64  
 1   Unnamed: 1            0 non-null      float64
 2   Name                  100 non-null    object 
 3   Profile ID            100 non-null    int64  
 4   Date                  100 non-null    object 
 5   Likes                 100 non-null    int64  
 6   Live video timestamp  100 non-null    object 
 7   Comment               99 non-null     object 
 8   Image URLs            1 non-null      object 
 9   Comment Permalink     0 non-null      float64
 10  Depth                 100 non-null    int64  
 11  Mentions              0 non-null      float64
 12  Reactions             0 non-null      float64
 13  Author Avatar         100 non-null    object 
 14  Comment Edited        1 non-null      float64
 15  Comment